In [5]:
import os
from brapi import Brapi
from dotenv import load_dotenv
import pandas as pd
import requests
api_k = 'qyWJMQp7dX6TcaKH88xyiM'
client = Brapi(
    api_key=api_k,
    environment="production",  # 'production' ou 'sandbox'
    max_retries=2,  # Número de tentativas (padrão: 2)
    timeout=60.0,  # Timeout em segundos (padrão: 60)
)

In [6]:
lista_ativos = []
caminho = os.path.join('magic_formula')
itens = os.listdir(caminho)
# print(itens)
for filename in itens:
    
    lista_ativos.append(filename)

In [7]:
print("Tamanho da lista de ativos: ",len(lista_ativos))
# lista_ativos

Tamanho da lista de ativos:  78


In [78]:
import time

dict_close, dict_adj, falhas = {}, {}, []

for ativo in lista_ativos:
    try:
        url = (f"https://brapi.dev/api/v2/stocks/historical?symbols={ativo}"
               f"&interval=1d&startDate=2015-01-01&endDate=2026-07-20&sortOrder=asc")
        r = requests.get(url, headers={"Authorization": api_k})
        dados = r.json()['results'][0]['data']['historicalDataPrice']

        df = pd.DataFrame(dados)
        df['date'] = pd.to_datetime(df['date'], unit='s').dt.normalize()  # timestamp -> data (zera as 03:00)
        df = df.set_index('date')
        df = df[~df.index.duplicated(keep='last')]                        # protege contra data repetida

        dict_close[ativo] = df['close']
        dict_adj[ativo]   = df['adjustedClose']
        time.sleep(0.3)                                                   # educação com a API
    except Exception as e:
        falhas.append(ativo)
        print(f"Ticker com problema: {ativo} -> {e}")

df_close = pd.DataFrame(dict_close).sort_index()   # linhas = datas, colunas = ativos, célula = fechamento
df_adj   = pd.DataFrame(dict_adj).sort_index()
print(df_close.shape, "| falhas:", falhas)


(2864, 78) | falhas: []


In [80]:
df_retornos = df_adj.pct_change()


In [83]:
df_retornos = df_retornos.dropna(how='all')

In [89]:
df_retornos.to_csv('retornos/retornos.csv')

In [90]:
df_retornos

,ABEV3,ALOS3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,...,TAEE11,TEND3,TOTS3,UGPA3,USIM5,VALE3,VBBR3,VIVT3,WEGE3,YDUQ3
date,,,,,,,,,,,,,,,,,,,,,
2015-01-05,-0.021863,NaN,-0.062594,-0.019441,-0.029455,-0.029493,-0.020742,-0.012326,0.002056,-0.031499,...,0.006441,NaN,-0.030726,-0.024677,-0.060554,-0.015029,NaN,-0.019833,-0.005811,NaN
2015-01-06,0.028737,NaN,-0.141497,0.009026,-0.019570,0.009745,0.013974,0.026667,0.032897,0.038968,...,0.009591,NaN,-0.054759,-0.006272,0.048899,0.040070,NaN,-0.024812,-0.015218,NaN
2015-01-07,0.024826,NaN,-0.047777,0.035704,0.010184,0.042981,0.044016,0.044159,0.039744,-0.011748,...,-0.005806,NaN,-0.014933,0.026474,0.059333,0.036695,NaN,0.034297,-0.001632,NaN
2015-01-08,0.004024,NaN,-0.007767,-0.018971,-0.004839,-0.010302,0.003414,0.002209,0.005145,-0.004622,...,0.009555,NaN,0.022902,-0.003170,-0.050009,0.010628,NaN,0.040662,0.012846,NaN
2015-01-09,-0.000610,NaN,0.003528,-0.021069,-0.010942,-0.021831,-0.043302,-0.035024,-0.043416,-0.028508,...,-0.023140,NaN,-0.010290,-0.014731,-0.040008,-0.021459,NaN,-0.015944,-0.005849,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-14,-0.001263,0.004625,0.017730,0.004773,-0.019251,0.013889,0.017292,-0.003674,-0.007459,0.002731,...,-0.002177,0.001447,-0.017112,-0.026511,-0.017900,0.015923,0.016484,0.022747,-0.004280,0.009009
2026-07-15,-0.015180,-0.000354,-0.327526,-0.041991,-0.010080,0.023483,-0.001943,0.003073,-0.001610,0.007923,...,-0.008240,-0.045087,0.041783,0.032879,-0.003645,0.006756,0.013514,-0.001408,0.001357,-0.022321
2026-07-16,0.001927,-0.015232,0.093264,-0.013685,-0.006967,-0.019120,0.010219,-0.020221,-0.010215,0.011545,...,0.002688,-0.016646,-0.000668,0.028617,-0.036585,-0.020534,0.018370,0.000000,-0.017397,0.009132
